In [ ]:
# ==========================================
# 1 Install / Upgrade libraries
# ==========================================
!pip install -U datasets accelerate bitsandbytes evaluate bert-score rouge-score pandas openpyxl scikit-learn
!pip install transformers==4.40.2
!pip install peft==0.4.0

# ==========================================
# 2 Imports
# ==========================================
import torch
import math
import time
import pandas as pd
import evaluate

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model
from google.colab import drive
from sklearn.model_selection import train_test_split

# ==========================================
# 3 Mount Google Drive
# ==========================================
drive.mount('/content/drive')

# ==========================================
# 4 Load Excel datasets
# ==========================================
phish = pd.read_excel("/content/drive/MyDrive/Realphishing.xlsx")
safe  = pd.read_excel("/content/drive/MyDrive/Realsafe.xlsx")

data = pd.concat([phish, safe], ignore_index=True)

# ==========================================
# 5 Fix NaN / datatype issues
# ==========================================
data = data.fillna("")
data["subject"] = data["subject"].astype(str)
data["body"]    = data["body"].astype(str)
data["label"]   = data["label"].astype(str)

# ==========================================
# 6 Convert to LLM instruction format
# ==========================================
data["text"] = (
    "Instruction: Generate a " + data["label"] + " email.\n"
    + "Subject: " + data["subject"] + "\n"
    + "Body: " + data["body"]
)

# ==========================================
# 7 Train / Validation split
# ==========================================
train_df, val_df = train_test_split(data, test_size=0.1, random_state=42)

train_dataset = Dataset.from_pandas(train_df)
val_dataset   = Dataset.from_pandas(val_df)

# ==========================================
# 8 Load TinyLlama model
# ==========================================
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16
)

# ==========================================
# 9 Apply LoRA (CHANGED: r=32, lora_alpha=64)
# ==========================================
config = LoraConfig(
    r=32,                    # increased from 16 → 32
    lora_alpha=64,           # increased from 32 → 64
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.03,
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, config)
model.print_trainable_parameters()

# ==========================================
# 10 Tokenization
# ==========================================
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset   = val_dataset.map(tokenize, batched=True)

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])
val_dataset.set_format(type="torch",   columns=["input_ids", "attention_mask"])

data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

# ==========================================
# 11 Training configuration (CHANGED)
# ==========================================
SAVE_PATH = "/content/drive/MyDrive/tinyllama_email_generator_final"

use_fp16 = torch.cuda.is_available()

training_args = TrainingArguments(
    output_dir=SAVE_PATH,
    per_device_train_batch_size=4,   # increased from 2 → 4
    per_device_eval_batch_size=4,    # increased from 2 → 4
    num_train_epochs=10,             # increased from 9 → 10
    learning_rate=1e-4,              # reduced from 1.5e-4 → 1e-4
    weight_decay=0.003,              # reduced from 0.005 → 0.003
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_strategy="epoch",
    report_to="none",
    fp16=use_fp16                    # auto-detects GPU
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

# ==========================================
# 12 Train model
# ==========================================
start = time.time()
trainer.train()
end = time.time()

print("Training time (minutes):", (end - start) / 60)

# ==========================================
# 13 Save best model
# ==========================================
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("Best model saved to:", SAVE_PATH)
print("Best checkpoint:", trainer.state.best_model_checkpoint)

# ==========================================
# 14 Perplexity (correct - ignores padding)
# ==========================================
model.eval()

total_loss = 0
count      = 0

with torch.no_grad():
    for sample in val_dataset.select(range(100)):
        input_ids      = sample["input_ids"].unsqueeze(0).to(model.device)
        attention_mask = sample["attention_mask"].unsqueeze(0).to(model.device)

        labels = input_ids.clone()
        labels[attention_mask == 0] = -100

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        total_loss += outputs.loss.item()
        count += 1

val_loss   = total_loss / count
perplexity = math.exp(val_loss)

# ==========================================
# 15 Generate sample predictions
# ==========================================
predictions = []
references  = []

for sample in val_dataset.select(range(50)):
    input_ids = sample["input_ids"].unsqueeze(0).to(model.device)

    with torch.no_grad():
        output = model.generate(
            input_ids=input_ids,
            max_new_tokens=120,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = tokenizer.decode(output[0], skip_special_tokens=True)
    reference = tokenizer.decode(sample["input_ids"], skip_special_tokens=True)

    predictions.append(generated)
    references.append(reference)

# ==========================================
# 16 ROUGE Score
# ==========================================
rouge       = evaluate.load("rouge")
rouge_score = rouge.compute(predictions=predictions, references=references)

# ==========================================
# 17 BERTScore
# ==========================================
bertscore  = evaluate.load("bertscore")
bert_score = bertscore.compute(
    predictions=predictions,
    references=references,
    lang="en",
    model_type="distilbert-base-uncased"
)

# ==========================================
# 18 Print all results
# ==========================================
bert_f1 = sum(bert_score["f1"])        / len(bert_score["f1"])
bert_p  = sum(bert_score["precision"]) / len(bert_score["precision"])
bert_r  = sum(bert_score["recall"])    / len(bert_score["recall"])

print("\n" + "=" * 50)
print("        EVALUATION RESULTS")
print("=" * 50)
print(f"\n  Validation Loss      : {val_loss:.4f}")
print(f"  Perplexity           : {perplexity:.4f}")
print(f"\n  ROUGE-1              : {rouge_score['rouge1']:.4f}")
print(f"  ROUGE-2              : {rouge_score['rouge2']:.4f}")
print(f"  ROUGE-L              : {rouge_score['rougeL']:.4f}")
print(f"\n  BERTScore Precision  : {bert_p:.4f}")
print(f"  BERTScore Recall     : {bert_r:.4f}")
print(f"  BERTScore F1         : {bert_f1:.4f}")
print("=" * 50)

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.5/527.5 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 18.6 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=f988d2dfba107f0f56c954e37be5fcb5b7e3f8324c569d4197947029be30c326
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score
  Attempting uninstall: pyarro

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]


        EVALUATION RESULTS

  Validation Loss      : 1.7582
  Perplexity           : 5.8021

  ROUGE-1              : 0.8862
  ROUGE-2              : 0.8861
  ROUGE-L              : 0.8868

  BERTScore Precision  : 0.9350
  BERTScore Recall     : 0.9934
  BERTScore F1         : 0.9613


In [ ]:
# ==========================================
# 1 Install libraries
# ==========================================
!pip install transformers==4.40.2
!pip install peft==0.4.0

# ==========================================
# 2 Imports
# ==========================================
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, PeftConfig

# ==========================================
# 3 Mount Drive
# ==========================================
from google.colab import drive
drive.mount('/content/drive')

# ==========================================
# 4 Load saved model
# ==========================================
SAVE_PATH = "/content/drive/MyDrive/tinyllama_email_generator_final"

peft_config = PeftConfig.from_pretrained(SAVE_PATH)
tokenizer   = AutoTokenizer.from_pretrained(SAVE_PATH, local_files_only=True)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    peft_config.base_model_name_or_path,
    device_map="auto",
    torch_dtype=torch.float16
)

model = PeftModel.from_pretrained(base_model, SAVE_PATH, local_files_only=True)
model.eval()
print("✅ Model loaded!")

# ==========================================
# 5 Email generation function
# ==========================================
def generate_email(label, subject, max_new_tokens=200):
    """
    label   : "phishing" or "safe"
    subject : email subject line
    """
    prompt = (
        f"Instruction: Generate a {label} email.\n"
        f"Subject: {subject}\n"
        f"Body:"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=max_new_tokens,
            do_sample=True,        # creative generation
            temperature=0.7,       # controls randomness (lower=focused, higher=creative)
            top_p=0.9,             # nucleus sampling
            top_k=50,              # limits vocab choices
            repetition_penalty=1.2, # avoids repeating phrases
            pad_token_id=tokenizer.eos_token_id
        )

    # decode only the newly generated tokens
    generated = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return generated.strip()

# ==========================================
# 6 Generate emails
# ==========================================

# --- Phishing email ---
print("=" * 60)
print("PHISHING EMAIL")
print("=" * 60)
phishing_email = generate_email(
    label="phishing",
    subject="Urgent: Your account has been compromised"
)
print(phishing_email)

# --- Safe email ---
print("\n" + "=" * 60)
print("SAFE EMAIL")
print("=" * 60)
safe_email = generate_email(
    label="safe",
    subject="Meeting rescheduled to Friday"
)
print(safe_email)

# ==========================================
# 7 Generate your own custom email
# ==========================================
print("\n" + "=" * 60)
print("CUSTOM EMAIL")
print("=" * 60)

my_label   = "phishing"   # change to "safe" or "phishing"
my_subject = "Your package could not be delivered"  # change to any subject

custom_email = generate_email(label=my_label, subject=my_subject)
print(custom_email)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 51.6 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.6.0
    Uninstalling huggingface_hub-1.6.0:
      Successfully uninstalled huggingface_hub-1.6.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the follow

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

✅ Model loaded!
PHISHING EMAIL
Dear Customer,We have recently identified unusual activity on your PayPal account.To prevent unauthorized access to your account and protect you from financial losses due to fraudulent transactions, we are temporarily suspending your ability to use your account until you verify the details listed below with us via our website or mobile app.Please follow the instructions provided by PayPal to complete the verification process so that your account can be reinstated.Thank you for your prompt attention to this matter.Best regards,PayPal© 2018All rights reserved."Terms of Use" "Privacy Policy" Contact Us Help Center Sign Out Log in Now Need help? Ask now Cancel Subscription Renewal Thank You for being part of the community!Your friends, family, colleagues, and more."Follow @paypal_us" Follow @paypal_us_en" Investor Relations Press Releases Corporate Information Job Opport

SAFE EMAIL
Dear students, the meeting with your academic advisor has been postponed unti